In [0]:
# Configuration
from pyspark.sql import functions as F, Window

dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("bronze_schema", "bronze", ["bronze", "gabrielajaniszews786_bronze"], "Bronze schema")
dbutils.widgets.combobox("silver_schema", "silver", ["silver", "gabrielajaniszews786_silver"], "Silver schema")
dbutils.widgets.dropdown("run_checks", "true", ["true", "false"], "Verification / experimentation")
RUN_CHECKS = dbutils.widgets.get("run_checks") == "true"
CATALOG       = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

BRONZE_SENSOR = f"{CATALOG}.{BRONZE_SCHEMA}.sensor_data"
VALID_SILVER_SENSOR = f"{CATALOG}.{SILVER_SCHEMA}.valid_sensor"
CHECKED_SILVER_SENSOR = f"{CATALOG}.{SILVER_SCHEMA}.checked_sensor"
QUARANTINE_SILVER_SENSOR = f"{CATALOG}.{SILVER_SCHEMA}.quarantine_sensor"




In [0]:
display(spark.sql(f"SELECT * FROM {BRONZE_SENSOR} LIMIT 10"))

In [0]:
source_df = spark.sql(f"SELECT * FROM {BRONZE_SENSOR}")
display(source_df.count())


In [0]:
checked_df = spark.sql(f"SELECT * FROM {CHECKED_SILVER_SENSOR}")
display(checked_df.count())
valid_df = spark.sql(f"SELECT * FROM {VALID_SILVER_SENSOR}")
display(valid_df.count())
quarantine_df = spark.sql(f"SELECT * FROM {QUARANTINE_SILVER_SENSOR}")
display(quarantine_df.count())

In [0]:
def reconcile(bronze_df, valid_df, quarantine_df):
    bronze_count = bronze_df.count()
    valid_count = valid_df.count()
    quarantine_count = quarantine_df.count()
    print(f"Bronze: {bronze_count:,}")
    print(f"Valid: {valid_count:,}")
    print(f"Quarantine: {quarantine_count:,}")
    print(f"Total: {bronze_count + valid_count + quarantine_count:,}")
    return bronze_count + valid_count + quarantine_count

assert reconcile(source_df,

    source_n, target_n = source_keys.count(), target_keys.count()
    missing_target_n = missing_in_target.count()
    missing_source_n = missing_in_source.count()

    print(f"\n=== {name} ===")
    print(f"  source keys: {source_n:,}")
    print(f"  target keys: {target_n:,}")
    print(f"  in source but not target: {missing_target_n:,}")
    print(f"  in target but not source: {missing_source_n:,}")

    # Persist so the metric can be tracked over time and queried by an alert
    (spark.createDataFrame(
        [(name, source_name, target_name,
          source_n, target_n, missing_target_n, missing_source_n)],
        "check_name string, source_table string, target_table string, "
        "source_keys long, target_keys long, missing_in_target long, missing_in_source long")
     .withColumn("checked_at", F.lit(check_ts).cast("timestamp"))
     .write.format("delta").mode("append").saveAsTable(results_table))

    return missing_in_target, missing_in_source